<a href="https://colab.research.google.com/github/NazmulHudaNabil/email-spam-detection/blob/main/Spam_or_not_using_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch

In [ ]:
df = pd.read_csv("/content/spam.csv")
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df.shape

(5157, 2)

In [ ]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.drop_duplicates(inplace=True)

### Target Value Lable Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
y = encoder.fit_transform(df['Category'])

In [ ]:
y

array([0, 0, 1, ..., 0, 0, 0])

### Data Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['Message'], y, test_size=0.1, random_state=42)

In [ ]:
X_train.shape, X_test.shape

((4641,), (516,))

### Create pytorch CustomDataset for Transformer

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# ---------------------------------------------------------
# 1. Define your custom dataset
# ---------------------------------------------------------
class MyTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx]) # Use .iloc for positional indexing
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )


        # Squeeze to remove batch dimension added by return_tensors="pt"
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

### Transformer Fine tuning "roberta-base" becasue it's good for Binary Classifcation

In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
train_dataset = MyTextDataset(X_train, y_train, tokenizer)
test_dataset = MyTextDataset(X_test, y_test, tokenizer)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

### Transformer Model training

In [ ]:
epochs = 5
for epoch in range(epochs):
    total_loss = 0

    for batch in train_loader:
        # Move batch to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        # Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} | Loss: {avg_loss:.4f}")

Epoch 1/5 | Loss: 0.0945
Epoch 2/5 | Loss: 0.0723
Epoch 3/5 | Loss: 0.0368
Epoch 4/5 | Loss: 0.0228
Epoch 5/5 | Loss: 0.0093


### Evalute the Test Data

In [ ]:
# Set the model to evaluation mode
model.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get predicted labels
        _, predicted = torch.max(logits, 1)

        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9942


#### Find model accuracy on test data is 99.42% it's very good

## Save this Model

In [ ]:
save_path = "./my_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./my_model/tokenizer_config.json', './my_model/tokenizer.json')

## Load the Model

In [ ]:
import os
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Use absolute path
save_path = "./my_model" # The model and tokenizer were saved directly to /content

tokenizer = RobertaTokenizer.from_pretrained(save_path)
model = RobertaForSequenceClassification.from_pretrained(save_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print("Loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded successfully!


## After the load the model again check Model accucay on test data becasuse to check Save & load worked correctly or not

In [ ]:
# Set the model to evaluation mode
model.eval()

correct_predictions = 0
total_predictions = 0

with torch.no_grad(): # Disable gradient calculation during evaluation
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        # Get predicted labels
        _, predicted = torch.max(logits, 1)

        total_predictions += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Test Accuracy: {accuracy:.4f}")

Test Accuracy: 0.9942


### Testing How it's work

In [ ]:
import torch

model.eval()

text = input("Enter Your text: ")
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)

# Move input tensors to the same device as the model
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=1).item()

spam_or_ham = "spam" if predicted_class == 1 else "ham"
print(f"Predicted class: {spam_or_ham}")

Enter Your text: Click this link. you will prize 100000 usd money
Predicted class: spam
